# 05 · Homework: apply DBSCAN to our dataset and test it against the other methods

**Lecture 2 · Block B10 · Homework · group work (suggested effort: 3-4 hours)**

**Learning goal:** We cluster our own dataset with the methods from the labs, apply DBSCAN ourselves, compare DBSCAN with the other methods and test whether our clusters are structure or artefact. Explaining clusters (profiles, surrogate trees) is not part of this homework.

**What we do:**
1. Task 1 - k-medoids (or k-means) and agglomerative clustering with two linkages on our D
2. Task 2 - **apply DBSCAN:** choose min_samples, read ε off the k-distance plot, report clusters and noise
3. Task 3 - **compare:** silhouette for all, ARI with the known groups, ARI between DBSCAN and each other method
4. Task 4 - **structure or artefact?** Repeat with another ε, a changed transformation, a different seed and an 80 % subsample

**Deliverable:** this notebook, executed from top to bottom, plus a one-page summary (template at the end). The easy part is running the algorithms; the tricky part is Tasks 3 and 4: where DBSCAN agrees with the other methods, why it differs, and which clusters survive.

In [ ]:
DATASET = "penguins"  # or "wholesale", "auto_mpg", "students" (see 01_dataset.ipynb)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.cluster.hierarchy import fcluster
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

import ddm

GROUP = "..."  # names of the group members
QUESTION = "..."  # our question (from 01_dataset.ipynb)

## 0 · Our toolbox

Paste our own, tested implementations from the lab notebooks into the next cell
(`preprocess`, `gower_distance` from 02, `agglomerative`, `k_medoids` from 03). Alternative:
put them into a file `notebooks/lecture02/my_toolbox.py` (next to this notebook) and write `from my_toolbox import *`.

In [ ]:
# TODO: replace these placeholders by our implementations from 02_distance and 03_clustering


def preprocess(df, types):
    raise NotImplementedError("paste preprocess from 02_distance.ipynb")


def gower_distance(df, types):
    raise NotImplementedError("paste gower_distance from 02_distance.ipynb")


def agglomerative(D, linkage="average"):
    raise NotImplementedError("paste agglomerative from 03_clustering.ipynb")


def k_medoids(D, k, seed=0, max_iter=100):
    raise NotImplementedError("paste k_medoids from 03_clustering.ipynb")

Given helpers from the lab notebooks (no need to change them):

In [ ]:
def keep_rows(df: pd.DataFrame, types: dict[str, str], min_present: float = 0.5) -> pd.DataFrame:
    """Drop objects where less than ``min_present`` of the used attributes are known."""
    used = [c for c, t in types.items() if t != "ignore"]
    share_present = df[used].notna().mean(axis=1)
    return df.loc[share_present >= min_present]

def log_transform(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Return a copy of df with log(1 + x) applied to the given columns."""
    out = df.copy()
    for col in columns:
        out[col] = np.log1p(out[col].astype(float))
    return out

def medoid_cost(D: np.ndarray, labels: np.ndarray, medoids: np.ndarray) -> float:
    """Sum of distances of every object to the medoid of its cluster."""
    return float(D[np.arange(len(labels)), medoids[labels]].sum())

def best_k_medoids(D: np.ndarray, k: int, n_init: int = 10, seed: int = 0):
    """Run k_medoids with several seeds and keep the result with the lowest cost."""
    best = None
    for run in range(n_init):
        labels, medoids = k_medoids(D, k, seed=seed + run)
        cost = medoid_cost(D, labels, medoids)
        if best is None or cost < best[2]:
            best = (labels, medoids, cost)
    return best


def evaluate(D: np.ndarray, data: pd.DataFrame, labels, label_candidates) -> dict:
    """Silhouette on D (without DBSCAN noise) and ARI against every label candidate."""
    labels = np.asarray(labels)
    clustered = labels != -1
    row = {"clusters": len(set(labels[clustered].tolist())), "noise": int((~clustered).sum())}
    if row["clusters"] >= 2:
        row["silhouette"] = silhouette_score(D[np.ix_(clustered, clustered)], labels[clustered], metric="precomputed")
    else:
        row["silhouette"] = np.nan
    for candidate in label_candidates:
        row[f"ARI vs {candidate}"] = adjusted_rand_score(data[candidate].fillna("missing").astype(str).to_numpy(), labels)
    return row



def k_distance_plot(D: np.ndarray, min_samples: int = 5, eps=None):
    """Sorted distance of every object to its (min_samples - 1)th nearest neighbour (itself excluded).

    Works for any distance matrix D. Put eps at the knee, where the curve starts to rise steeply.
    """
    kd = np.sort(np.sort(D, axis=1)[:, min_samples - 1])
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(kd)
    if eps is not None:
        ax.axhline(eps, color="red", linestyle="--", label=f"eps = {eps}")
        ax.legend()
    ax.set_xlabel("objects, sorted by their k-distance")
    ax.set_ylabel(f"distance to neighbour no. {min_samples - 1}")
    plt.show()
    return kd


def ari_between(results: dict) -> pd.DataFrame:
    """ARI between every pair of clusterings (DBSCAN noise counts as its own group)."""
    names = list(results)
    return pd.DataFrame([[adjusted_rand_score(results[a], results[b]) for b in names] for a in names],
                        index=names, columns=names).round(3)

## 1 · Data and transformation

Load the data, use our TYPES (saved by 01, or define them here), choose log columns,
compute `X` with `preprocess` and `D` with `gower_distance` (or another distance we can justify).

In [ ]:
df = ddm.load_dataset(DATASET)
info = ddm.dataset_info(DATASET)
TYPES = ddm.load_json(DATASET, "types")  # from 01_dataset.ipynb, or define the dict here
LOG_COLUMNS = []

# TODO: data = log_transform(keep_rows(df, TYPES), LOG_COLUMNS); X = ...; D = ...
raise NotImplementedError("Section 1: compute data, X and D with our own functions")

## 2 · Task 1: clusterings from the labs

Collect every clustering as an array of labels in the dict `results`, with a descriptive name
(algorithm and parameters). Choose k with the elbow and the silhouette (see 04_beyond).

In [ ]:
results = {}
# TODO, for example:
# results["k-medoids (k=3)"] = best_k_medoids(D, 3)[0]
# results["average linkage (k=3)"] = fcluster(agglomerative(D, "average"), 3, "maxclust") - 1
# results["complete linkage (k=3)"] = ...
raise NotImplementedError("Task 1: run the clusterings")

## 3 · Task 2: apply DBSCAN

1. Choose `min_samples` (at least 4; larger for noisy data).
2. Draw the k-distance plot of **our** D with `k_distance_plot(D, min_samples)` and read ε off
   the knee (the same D that DBSCAN gets, whatever distance it is).
3. Run DBSCAN on the precomputed D, add the result to `results` and report the number of
   clusters and of noise points. Look at the result in an MDS (or PCA) plot.

In [ ]:
# TODO, for example:
# MIN_SAMPLES = 5
# kd = k_distance_plot(D, MIN_SAMPLES)
# EPS = ...  # read off the knee
# k_distance_plot(D, MIN_SAMPLES, eps=EPS)
# db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, metric="precomputed").fit_predict(D)
# results[f"DBSCAN (eps={EPS}, min_samples={MIN_SAMPLES})"] = db
# print("clusters:", db.max() + 1, " noise:", (db == -1).sum())
# ddm.plot_clusters_mds(D, {"DBSCAN": db})
raise NotImplementedError("Task 2: DBSCAN with eps from the k-distance plot")

*Our choice (2-3 sentences): why this min_samples, where is the knee, which objects are noise?*

## 4 · Task 3: DBSCAN against the other methods

In [ ]:
# TODO: table = pd.DataFrame({name: evaluate(D, data, lab, info.label_candidates) for name, lab in results.items()}).T
#       display(table); display(ari_between(results))
#       plus one picture: ddm.plot_clusters_mds(D, results) or ddm.plot_clusters_pca(X, results)
raise NotImplementedError("Task 3: evaluation table, ARI between the methods, plot")

*Our interpretation (4-6 sentences): where does DBSCAN agree with the other methods (ARI between
them), where not, and why (shape, density, noise, number of clusters)? Which clustering is best by
the silhouette, which by the ARI with the known groups, and why do they disagree?*

## 5 · Task 4: structure or artefact?

A cluster that only exists for one seed, one scaling or one particular sample is probably an
**artefact** of our choices. A cluster that survives changes is a candidate for real
**structure**. We repeat DBSCAN and our best other clustering with one change at a time:

| Change | Example |
|--------|---------|
| ε (DBSCAN) | 0.8 × and 1.2 × our ε: which clusters merge, split or turn into noise? |
| transformation | other scaling, log or no log, other distance (Euclidean on X instead of Gower), drop one attribute |
| seed | `seed=1, 2, ...` instead of 0 (for k-medoids / k-means) |
| 80 % subsample | cluster a random 80 % of the objects again and compare on these objects |

**How we compare:** for every cluster of the original clustering we look for the cluster of
the new clustering with the largest overlap (Jaccard index = |A ∩ B| / |A ∪ B|).
Rule of thumb (Hennig 2007): Jaccard ≥ 0.75 → the cluster *survives*; below 0.5 → it *dissolves*.
The ARI between the two clusterings gives one overall number.

### Helper sketch (given) and a demo on the village map

In [ ]:
def cluster_overlap(base_labels, other_labels) -> pd.DataFrame:
    """For every cluster of base: best-matching cluster of other (same objects!) and Jaccard index."""
    base = np.asarray(base_labels)
    other = np.asarray(other_labels)
    rows = []
    for c in np.unique(base[base != -1]):
        in_c = base == c
        best_match, best_jaccard = None, 0.0
        for d in np.unique(other[other != -1]):
            in_d = other == d
            jaccard = (in_c & in_d).sum() / (in_c | in_d).sum()
            if jaccard > best_jaccard:
                best_match, best_jaccard = d, jaccard
        verdict = "survives" if best_jaccard >= 0.75 else ("unclear" if best_jaccard >= 0.5 else "dissolves")
        rows.append({"cluster": c, "size": int(in_c.sum()), "best match": best_match,
                     "Jaccard": round(best_jaccard, 2), "verdict": verdict})
    return pd.DataFrame(rows).set_index("cluster")


def subsample(n: int, fraction: float = 0.8, seed: int = 0) -> np.ndarray:
    """Sorted random positions of a subsample (without replacement)."""
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(n, size=int(round(fraction * n)), replace=False))


# Demo: k-means (k=5) on the village map
village = ddm.load_toy_village()
xy = village[["x", "y"]].to_numpy()
base = KMeans(n_clusters=5, n_init=10, random_state=0).fit_predict(xy)

# (a) different seed (and only one start, to make the effect visible)
other_seed = KMeans(n_clusters=5, n_init=1, random_state=3).fit_predict(xy)
print("(a) different seed: ARI", round(adjusted_rand_score(base, other_seed), 3))
display(cluster_overlap(base, other_seed))

# (b) 80 % subsample: cluster only the subsample, compare on the same objects
idx = subsample(len(xy), 0.8, seed=0)
sub = KMeans(n_clusters=5, n_init=10, random_state=0).fit_predict(xy[idx])
print("(b) 80 % subsample: ARI on the common objects", round(adjusted_rand_score(base[idx], sub), 3))
display(cluster_overlap(base[idx], sub))

# (c) changed transformation: the river counts as 3 km
xy_river = np.column_stack([xy, 3.0 * (village["bank"] == "east").to_numpy()])
river = KMeans(n_clusters=5, n_init=10, random_state=0).fit_predict(xy_river)
print("(c) changed transformation (river): ARI", round(adjusted_rand_score(base, river), 3))
display(cluster_overlap(base, river))

### Our sensitivity analysis

In [ ]:
# TODO: BEST_LABELS = results[BEST]  (and the DBSCAN labels)
# (0) DBSCAN with 0.8 * EPS and 1.2 * EPS, cluster_overlap(...) against our DBSCAN result
# (a) changed transformation: recompute X / D with another choice, cluster again, cluster_overlap(...)
# (b) different seed: cluster again with seed=1 (and 2, 3 ...), cluster_overlap(...)
# (c) 80 % subsample: idx = subsample(len(D), 0.8, seed=0); cluster D[np.ix_(idx, idx)] again,
#     compare with BEST_LABELS[idx]; repeat for a few seeds
raise NotImplementedError("Task 4: sensitivity analysis")

*Summary of Task 4:* which clusters survive the changes, which dissolve? Which of the
surviving clusters have a domain meaning, and which are probably artefacts (of scaling, of
one attribute, of a few outliers)?

---

## One-page summary (template)

**Group, dataset, question:** ...

**Data and transformation:** objects, attributes used and their types, missing values,
scaling / log, distance. One sentence why.

**Methods:** which algorithms with which parameters (k, linkage, ε, min_samples); how we read ε
off the k-distance plot.

**Results:** best clustering by silhouette and by ARI (numbers), one figure.

**DBSCAN against the others:** ARI between the methods, number of noise points; where DBSCAN
agrees with the other methods and why it differs.

**Structure or artefact?** which clusters survive transformation, seed and subsample (Jaccard
values); which ones have a domain meaning.

**Limitations and next step:** what we would do with more time.